# Train classification model

Imports

In [1]:
import torch
from torch import nn
import torchvision.transforms as transforms
import sys
import os
import torch.nn.functional as F
from ray.train import Checkpoint, get_checkpoint
from ray import tune

# Absolute path to your project's src folder
src_path = os.path.abspath(os.path.join("..", "src"))

# Add to sys.path if not already present
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from data import get_data_loaders, show_image
from models.classification_model import ClassificationModel
from parameter_optimization import optimize_parameters, load_model
import config

Data Initialization

In [2]:
trainloader, testloader = get_data_loaders(
    "Mirabest",
    transforms.Compose([
        transforms.ToTensor(), # to range [0,1]
        transforms.Normalize([0.5], [0.5]) # 0 centers
        ]),
        config.CLASSIFICATION_MODEL["batch_size"])

print(next(iter(trainloader))[0].shape)
print(next(iter(trainloader)))

Files already downloaded and verified
Files already downloaded and verified


torch.Size([2, 1, 150, 150])
[tensor([[[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]]],


        [[[-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          ...,
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.],
          [-1., -1., -1.,  ..., -1., -1., -1.]]]]), tensor([1, 1])]


Parameter optimization

In [3]:
model = ClassificationModel()

print_num = 10
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adagrad(model.parameters(), lr=0.01)

# Optimize parameters
config_obj = {
    'lr': tune.loguniform(1e-4, 1e-1),
    'optimizer': tune.choice([torch.optim.AdamW, torch.optim.Adam]),
    'model_class': ClassificationModel,
    'criterion_class': nn.CrossEntropyLoss,
    'dataset': 'MiraBest'
}

results = optimize_parameters(config_obj)
print(f"Best results: {results.get_best_result().config}")
for epoch in range(config.CLASSIFICATION_MODEL["num_epochs"]):  # loop over the dataset multiple times

    print(f'Epoch {epoch+1}')
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % print_num == (print_num-1):    # print every 50 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / print_num))
            running_loss = 0.0

print('Finished Training')



(TemporaryActor pid=12473) No module named 'models'
(TemporaryActor pid=12473) Traceback (most recent call last):
(TemporaryActor pid=12473)   File "/Users/bevanslabbert/Documents/GitHub/pid-radast/.venv/lib/python3.13/site-packages/ray/_private/serialization.py", line 458, in deserialize_objects
(TemporaryActor pid=12473)     obj = self._deserialize_object(data, metadata, object_ref)
(TemporaryActor pid=12473)   File "/Users/bevanslabbert/Documents/GitHub/pid-radast/.venv/lib/python3.13/site-packages/ray/_private/serialization.py", line 315, in _deserialize_object
(TemporaryActor pid=12473)     return self._deserialize_msgpack_data(data, metadata_fields)
(TemporaryActor pid=12473)            ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
(TemporaryActor pid=12473)   File "/Users/bevanslabbert/Documents/GitHub/pid-radast/.venv/lib/python3.13/site-packages/ray/_private/serialization.py", line 270, in _deserialize_msgpack_data
(TemporaryActor pid=12473)     python_objects = self._

(raylet) Traceback (most recent call last):
  File "python/ray/_raylet.pyx", line 1843, in ray._raylet.execute_task
  File "python/ray/_raylet.pyx", line 1877, in ray._raylet.execute_task
  File "python/ray/_raylet.pyx", line 979, in ray._raylet.raise_if_dependency_failed
ray.exceptions.RaySystemError: System error: No module named 'models'
traceback: Traceback (most recent call last):
  File "/Users/bevanslabbert/Documents/GitHub/pid-radast/.venv/lib/python3.13/site-packages/ray/_private/serialization.py", line 458, in deserialize_objects
    obj = self._deserialize_object(data, metadata, object_ref)
  File "/Users/bevanslabbert/Documents/GitHub/pid-radast/.venv/lib/python3.13/site-packages/ray/_private/serialization.py", line 315, in _deserialize_object
    return self._deserialize_msgpack_data(data, metadata_fields)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/bevanslabbert/Documents/GitHub/pid-radast/.venv/lib/python3.13/site-packages/ray/_private

2025-07-20 12:16:12,565	ERROR tune_controller.py:1331 -- Trial task failed for trial train_8a316_00002
Traceback (most recent call last):
  File "/Users/bevanslabbert/Documents/GitHub/pid-radast/.venv/lib/python3.13/site-packages/ray/air/execution/_internal/event_manager.py", line 110, in resolve_future
    result = ray.get(future)
  File "/Users/bevanslabbert/Documents/GitHub/pid-radast/.venv/lib/python3.13/site-packages/ray/_private/auto_init_hook.py", line 22, in auto_init_wrapper
    return fn(*args, **kwargs)
  File "/Users/bevanslabbert/Documents/GitHub/pid-radast/.venv/lib/python3.13/site-packages/ray/_private/client_mode_hook.py", line 104, in wrapper
    return func(*args, **kwargs)
  File "/Users/bevanslabbert/Documents/GitHub/pid-radast/.venv/lib/python3.13/site-packages/ray/_private/worker.py", line 2849, in get
    values, debugger_breakpoint = worker.get_objects(object_refs, timeout=timeout)
                                  ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

RuntimeError: No best trial found for the given metric: val_loss. This means that no trial has reported this metric, or all values reported for this metric are NaN. To not ignore NaN values, you can set the `filter_nan_and_inf` arg to False.

In [ ]:
optimized_model = ClassificationModel()
optimized_model = load_model(optimized_model, 'classification') # load optimized parameters

In [ ]:
# get data
dataiter = iter(testloader)
images, labels = next(dataiter)

# get predictions
output = model(images)
_, predicted = torch.max(output, 1)

In [ ]:
correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 88 test images: %d %%' % (100 * correct / total))

Accuracy of the network on the 88 test images: 75 %


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adagrad(optimized_model.parameters(), lr=0.01)

for epoch in range(config.CLASSIFICATION_MODEL["num_epochs"]):  # loop over the dataset multiple times
    print(f'Epoch {epoch+1}')
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # get the inputs
        inputs, labels = data

        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = optimized_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()
        if i % print_num == (print_num-1):    # print every 50 mini-batches
            print('[%d, %5d] loss: %.3f' %
                  (epoch + 1, i + 1, running_loss / print_num))
            running_loss = 0.0

print('Finished Training')

Epoch 1
[1,    10] loss: 0.763
[1,    20] loss: 0.822
[1,    30] loss: 0.059
[1,    40] loss: 0.552
[1,    50] loss: 0.145
[1,    60] loss: 0.120
[1,    70] loss: 0.609
[1,    80] loss: 0.200
[1,    90] loss: 0.384
[1,   100] loss: 1.356
[1,   110] loss: 0.514
[1,   120] loss: 0.105
[1,   130] loss: 0.193
[1,   140] loss: 0.037
[1,   150] loss: 0.162
[1,   160] loss: 0.291
[1,   170] loss: 0.025
[1,   180] loss: 0.088
[1,   190] loss: 0.662
[1,   200] loss: 0.043
[1,   210] loss: 0.164
[1,   220] loss: 0.194
[1,   230] loss: 0.048
[1,   240] loss: 0.277
[1,   250] loss: 0.233
[1,   260] loss: 0.142
[1,   270] loss: 0.142
[1,   280] loss: 0.257
[1,   290] loss: 0.178
[1,   300] loss: 0.499
[1,   310] loss: 0.524
[1,   320] loss: 0.323
[1,   330] loss: 0.169
[1,   340] loss: 0.407
Epoch 2
[2,    10] loss: 0.073
[2,    20] loss: 0.044
[2,    30] loss: 0.043
[2,    40] loss: 0.086
[2,    50] loss: 0.238
[2,    60] loss: 0.106
[2,    70] loss: 0.060
[2,    80] loss: 0.066
[2,    90] loss: 0

In [ ]:
# get data
dataiter = iter(testloader)
images, labels = next(dataiter)

# get predictions
output = optimized_model(images)
_, predicted = torch.max(output, 1)

correct = 0
total = 0
with torch.no_grad():
    for data in testloader:
        images, labels = data
        outputs = optimized_model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print('Accuracy of the network on the 88 test images: %d %%' % (100 * correct / total))


Accuracy of the network on the 88 test images: 74 %


Train

Test